# Автоэнкодер для поиска подозрительных наблюдений

## Цель

Эта тетрадь показывает второй нейросетевой демонстрационный блок: поиск строк, которые хуже всего восстанавливаются простой автоэнкодерной моделью. Цель — получить список наблюдений для дальнейшей проверки, а не автоматически исправлять или удалять данные.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from src.analysis.autoencoder_anomaly_detection import run_autoencoder_anomaly_detection

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
reports_dir = project_root / 'reports'


## Метод

Автоэнкодер — нейросетевая модель, которая учится восстанавливать входные признаки. В этой тетради используется `MLPRegressor` из `scikit-learn`: входная матрица признаков одновременно является и целевой матрицей для восстановления.

Используется тот же осторожный набор признаков, что и в базовом моделировании: целевая переменная, прямые метрики изменения береговой бровки, идентификаторы и служебные текстовые поля не подаются в модель как признаки.

In [ ]:
outputs = run_autoencoder_anomaly_detection(verbose=False)

pd.DataFrame({
    'показатель': [
        'обработано строк',
        'признаков до кодирования категорий',
        'признаков после предобработки',
        'строк в обучающей части для порога',
        'порог ошибки восстановления',
        'подозрительных строк',
    ],
    'значение': [
        outputs['n_rows'],
        outputs['n_features_raw'],
        outputs['n_features_encoded'],
        outputs['n_train'],
        round(outputs['threshold'], 6),
        outputs['n_anomalies'],
    ],
})

## Что считается подозрительным

Для каждой строки считается ошибка восстановления: средняя квадратичная разница между подготовленным вектором признаков и его восстановленной версией. Подозрительными считаются строки выше 95-го процентиля ошибки восстановления на обучающей части.

Такая строка не считается доказанной ошибкой. Она означает только, что сочетание признаков нетипично для текущей модели и требует предметной проверки.

In [ ]:
scores = pd.read_csv(reports_dir / 'tables' / 'autoencoder_anomaly_scores.csv')
top_anomalies = pd.read_csv(reports_dir / 'tables' / 'autoencoder_top_anomalies.csv')

display(Markdown((reports_dir / 'tables' / 'autoencoder_anomaly_summary.md').read_text(encoding='utf-8')))

## Результаты

Первый график показывает распределение ошибок восстановления и выбранный порог. Второй график показывает 25 строк с наибольшей ошибкой восстановления. Эти строки удобны как короткий список для ручной проверки.

In [ ]:
display(Image(filename=str(reports_dir / 'figures' / '04_autoencoder_reconstruction_error.png')))
display(Image(filename=str(reports_dir / 'figures' / '04_autoencoder_top_anomalies.png')))

In [ ]:
top_anomalies.head(25)

## Ограничения

- Это демонстрационный нейросетевой метод, а не финальное научное решение.
- Автоэнкодер не доказывает, что строка ошибочна; он только выделяет нетипичные сочетания признаков.
- Найденные строки нельзя автоматически удалять из датасета.
- Результат зависит от набора признаков, качества предобработки и покрытия исходных данных.
- Следующий шаг — сопоставить отмеченные строки с QC-пояснениями, конфликтующими дублями, участками и датами интервалов.